In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "openpyxl", "tqdm"])
import re
import os, random, time, copy
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR="/kaggle/working/"; os.makedirs(OUTPUT_DIR,exist_ok=True)

D=256; VOCAB_SIZE=30522; MAX_SEQ=64; HEADS=4; DROP=0.15; CBAM_BLOCKS=3; TEXT_ENC_LAYERS=2; TEXT_REFINE_LAYERS=2; FUSE_LAYERS=4
NUM_CLIENTS=5; ROUNDS=20; LOCAL_EP=3; BS=32; FED_LR=3e-4; WD=1e-4; MAX_ANS_VOCAB=300; MIN_ANS_FREQ=3

# ─────────────────────────────────────────────────────────────────────
# 4. VizWiz  (e.g., Eldon/VizWiz or any HF mirror)
# ─────────────────────────────────────────────────────────────────────
# ~31,000 QA pairs from blind users · real-world photos
# Highly noisy: unanswerable questions, poor image quality,
# answers from 10 crowd annotators. Very diverse open-ended answers
# (objects, colors, text reading, brands, counts, etc.)

def normalize_answer_vizwiz(ans: str) -> str:
    """Normalize VizWiz answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Unanswerable / unsuitable canonicalization ──
    unanswerable_set = {
        'unanswerable', 'unsuitable', 'unsuitable image',
        'not answerable', 'cant answer', 'cannot answer',
        'i dont know', 'i don\'t know', 'i do not know',
        'unable to answer', 'not sure', 'unclear',
        'unreadable', 'cannot be determined', 'cant be determined',
        'can not be determined', 'not clear', 'blurry',
        'too blurry', 'too dark', 'no answer', 'na', 'n/a',
        'cannot tell', 'cant tell', 'hard to tell',
        'impossible to tell', 'nothing', 'not possible',
    }
    if ans in unanswerable_set:
        return 'unanswerable'

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true',
               'yes it is', 'yes, it is', 'yea', 'ya'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'no it is not', 'nah'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric canonicalization ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10',
                   'eleven': '11', 'twelve': '12', 'thirteen': '13',
                   'fourteen': '14', 'fifteen': '15', 'twenty': '20',
                   'thirty': '30', 'forty': '40', 'fifty': '50',
                   'hundred': '100'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Color normalization (very common in VizWiz) ──
    color_map = {
        'blue': 'blue', 'light blue': 'blue', 'dark blue': 'blue',
        'navy': 'blue', 'navy blue': 'blue', 'royal blue': 'blue',
        'red': 'red', 'dark red': 'red', 'light red': 'red',
        'maroon': 'red', 'crimson': 'red', 'burgundy': 'red',
        'green': 'green', 'light green': 'green', 'dark green': 'green',
        'lime': 'green', 'olive': 'green',
        'yellow': 'yellow', 'light yellow': 'yellow', 'gold': 'yellow',
        'golden': 'yellow',
        'orange': 'orange',
        'pink': 'pink', 'light pink': 'pink', 'hot pink': 'pink',
        'magenta': 'pink',
        'purple': 'purple', 'violet': 'purple', 'lavender': 'purple',
        'brown': 'brown', 'tan': 'brown', 'beige': 'brown',
        'khaki': 'brown',
        'white': 'white', 'off white': 'white', 'cream': 'white',
        'ivory': 'white',
        'black': 'black', 'dark': 'black',
        'gray': 'gray', 'grey': 'gray', 'silver': 'gray',
        'light gray': 'gray', 'light grey': 'gray',
        'dark gray': 'gray', 'dark grey': 'gray',
    }
    if ans in color_map:
        return color_map[ans]

    # ── Common object synonyms ──
    object_map = {
        'cellphone': 'phone', 'cell phone': 'phone', 'mobile': 'phone',
        'mobile phone': 'phone', 'smartphone': 'phone', 'iphone': 'phone',
        'tv': 'television', 'television': 'television',
        'laptop': 'laptop', 'computer': 'laptop', 'notebook': 'laptop',
        'can': 'can', 'cans': 'can', 'tin': 'can',
        'bottle': 'bottle', 'bottles': 'bottle',
        'box': 'box', 'boxes': 'box', 'package': 'box',
        'shirt': 'shirt', 'tshirt': 'shirt', 't-shirt': 'shirt',
        't shirt': 'shirt', 'tee shirt': 'shirt',
        'pants': 'pants', 'trousers': 'pants', 'jeans': 'pants',
        'shoe': 'shoe', 'shoes': 'shoe', 'sneaker': 'shoe',
        'sneakers': 'shoe',
        'remote': 'remote', 'remote control': 'remote',
        'glasses': 'glasses', 'eyeglasses': 'glasses',
        'sunglasses': 'sunglasses',
        'soda': 'soda', 'pop': 'soda', 'soft drink': 'soda',
        'coke': 'coca cola', 'coca-cola': 'coca cola',
        'pepsi': 'pepsi', 'dr pepper': 'dr pepper',
        'cat': 'cat', 'cats': 'cat', 'kitten': 'cat',
        'dog': 'dog', 'dogs': 'dog', 'puppy': 'dog',
        'dollar': 'dollar', 'dollars': 'dollar',
        'cent': 'cent', 'cents': 'cent',
    }
    if ans in object_map:
        return object_map[ans]

    # ── Remove articles and fillers ──
    ans = re.sub(r'^(the|a|an|its|it is|this is|that is|it\'s|i think)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Remove trailing period ──
    ans = ans.rstrip('.')

    return ans

# ── LOAD VizWiz ──
print("\n"+"="*60+"\nLOADING VizWiz INTO RAM\n"+"="*60)
from datasets import load_dataset; ds=load_dataset('lmms-lab/VizWiz-VQA')
def majority_answer(answers):
    if not answers: return ''
    clean=[str(a.get('answer','') if isinstance(a,dict) else a).strip().lower() for a in answers]
    clean=[a for a in clean if a and a not in ('unanswerable','unsuitable')]
    return Counter(clean).most_common(1)[0][0] if clean else ''
all_samples=[]
for s in tqdm(ds['val'],desc="VizWiz val"):
    try:
        img=s.get('image'); q=str(s.get('question',''))
        if not img or not q: continue
        answer=s.get('answer')
        if answer is not None: answer=str(answer).strip().lower()
        else: answer=majority_answer(s.get('answers',[]))
        answer = normalize_answer_vizwiz(answer)
        if not answer or answer in ('unanswerable','unsuitable',''): continue
        all_samples.append({'image':np.array(img.convert('RGB').resize((224,224)),dtype=np.float32)/255.0,'question':q,'answer':answer})
    except: continue
del ds; random.shuffle(all_samples); sp=int(len(all_samples)*0.8)
train_samples,test_samples=all_samples[:sp],all_samples[sp:]
all_ans=[s['answer'] for s in all_samples]; counts=Counter(all_ans)
filtered=[(a,c) for a,c in counts.most_common() if c>=MIN_ANS_FREQ][:MAX_ANS_VOCAB]
answer_vocab={'<unk>':0}
for i,(a,_) in enumerate(sorted(filtered,key=lambda x:x[0])): answer_vocab[a]=i+1
num_classes=len(answer_vocab)
print(f"  Train:{len(train_samples)}, Test:{len(test_samples)}, Classes:{num_classes}")

def tokenize(qs):
    il,ml=[],[]
    for q in qs:
        w=q.lower().split()[:MAX_SEQ-2]; ids=[1]+[hash(x)%(VOCAB_SIZE-2)+2 for x in w]+[2]; m=[1.0]*len(ids)
        while len(ids)<MAX_SEQ: ids.append(0); m.append(0.0)
        il.append(ids[:MAX_SEQ]); ml.append(m[:MAX_SEQ])
    return il,ml

class VQADataset(Dataset):
    def __init__(s,samples,vocab,aug=False): s.samples=samples; s.vocab=vocab; s.aug=aug; s.ids,s.masks=tokenize([x['question'] for x in samples])
    def __len__(s): return len(s.samples)
    def __getitem__(s,i):
        x=s.samples[i]; img=torch.tensor(x['image']).permute(2,0,1)
        if s.aug and random.random()>0.5: img=img.flip(-1)
        return img,torch.tensor(s.ids[i],dtype=torch.long),torch.tensor(s.masks[i],dtype=torch.float32),s.vocab.get(x['answer'],0)
def collate_fn(b): i,d,m,l=zip(*b); return torch.stack(i),torch.stack(d),torch.stack(m),torch.tensor(l,dtype=torch.long)

idx=np.random.permutation(len(train_samples)); sz=len(train_samples)//NUM_CLIENTS
client_splits={c:idx[c*sz:(c+1)*sz if c<NUM_CLIENTS-1 else len(train_samples)].tolist() for c in range(NUM_CLIENTS)}
client_loaders,client_sizes={},{}
for cid,indices in client_splits.items():
    client_loaders[cid]=DataLoader(VQADataset([train_samples[i] for i in indices],answer_vocab,True),batch_size=BS,shuffle=True,num_workers=2,pin_memory=True,collate_fn=collate_fn)
    client_sizes[cid]=len(indices); print(f"  C{cid}: {len(indices)}")
test_loader=DataLoader(VQADataset(test_samples,answer_vocab),batch_size=BS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)

# ── MODEL (same as centralized) ──
class TransformerBlock(nn.Module):
    def __init__(s,dim,n_heads=4,ffn_ratio=4,dropout=0.1): super().__init__(); s.norm1=nn.LayerNorm(dim); s.norm2=nn.LayerNorm(dim); s.attn=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); s.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*ffn_ratio,dim),nn.Dropout(dropout))
    def forward(s,x,mask=None): h=s.norm1(x); kpm=(mask==0) if mask is not None else None; h,_=s.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+s.ffn(s.norm2(x))
class VisionEncoder(nn.Module):
    def __init__(s,dim=256): super().__init__(); s.c1=nn.Conv2d(3,32,7,2,3,bias=False); s.b1=nn.BatchNorm2d(32); s.p1=nn.MaxPool2d(3,2,1); s.c2=nn.Conv2d(32,64,3,2,1,bias=False); s.b2=nn.BatchNorm2d(64); s.c3=nn.Conv2d(64,128,3,2,1,bias=False); s.b3=nn.BatchNorm2d(128); s.c4=nn.Conv2d(128,dim,3,2,1,bias=False); s.b4=nn.BatchNorm2d(dim); s.norm=nn.LayerNorm(dim)
    def forward(s,x): h=s.p1(F.silu(s.b1(s.c1(x)))); h=F.silu(s.b2(s.c2(h))); h=F.silu(s.b3(s.c3(h))); h=F.silu(s.b4(s.c4(h))); B,C,H,W=h.shape; return s.norm(h.permute(0,2,3,1).reshape(B,H*W,C))
class TextEncoder(nn.Module):
    def __init__(s,vs=30522,dim=256,nl=2,nh=4,ml=64,do=0.1): super().__init__(); s.te=nn.Embedding(vs,dim); s.pe=nn.Parameter(torch.randn(1,ml,dim)*0.02); s.en=nn.LayerNorm(dim); s.ed=nn.Dropout(do); s.blocks=nn.ModuleList([TransformerBlock(dim,nh,dropout=do) for _ in range(nl)]); s.fn=nn.LayerNorm(dim)
    def forward(s,ids,mask=None): L=ids.shape[1]; x=s.te(ids)+s.pe[:,:L,:]; x=s.ed(s.en(x)); [x:=b(x,mask=mask) for b in s.blocks]; return s.fn(x)
class ChannelAttention(nn.Module):
    def __init__(s,ch,r=8): super().__init__(); s.f1=nn.Linear(ch,ch//r,bias=False); s.f2=nn.Linear(ch//r,ch,bias=False)
    def forward(s,x): a=x.mean([1,2],keepdim=True); m=x.amax([1,2],keepdim=True); return x*torch.sigmoid(s.f2(F.silu(s.f1(a)))+s.f2(F.silu(s.f1(m))))
class SpatialAttention(nn.Module):
    def __init__(s): super().__init__(); s.c1=nn.Conv2d(2,8,3,padding=1,bias=False); s.c2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False); s.fuse=nn.Conv2d(16,1,1,bias=False)
    def forward(s,x): xp=x.permute(0,3,1,2); a=xp.mean(1,keepdim=True); m=xp.amax(1,keepdim=True); c=torch.cat([a,m],1); return x*torch.sigmoid(s.fuse(torch.cat([s.c1(c),s.c2(c)],1))).permute(0,2,3,1)
class CBAMBlock(nn.Module):
    def __init__(s,ch): super().__init__(); s.ca=ChannelAttention(ch); s.sa=SpatialAttention(); s.ffn=nn.Sequential(nn.Linear(ch,ch*2),nn.GELU(),nn.Linear(ch*2,ch)); s.n1=nn.LayerNorm(ch); s.n2=nn.LayerNorm(ch)
    def forward(s,t): B,N,C=t.shape; sp=s.sa(s.ca(t.reshape(B,7,7,C))); t=s.n1(t+sp.reshape(B,N,C)); return s.n2(t+s.ffn(t))
class FusionLayer(nn.Module):
    def __init__(s,dim,nh,do): super().__init__(); s.v2t=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.v2tn=nn.LayerNorm(dim); s.v2tf=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim)); s.v2tfn=nn.LayerNorm(dim); s.t2v=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.t2vn=nn.LayerNorm(dim); s.t2vf=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim)); s.t2vfn=nn.LayerNorm(dim)
    def forward(s,v,t,kpm=None): o,_=s.v2t(v,t,t,key_padding_mask=kpm); v=s.v2tn(v+o); v=s.v2tfn(v+s.v2tf(v)); o,_=s.t2v(t,v,v); t=s.t2vn(t+o); t=s.t2vfn(t+s.t2vf(t)); return v,t

class MedicalVQAModel(nn.Module):
    def __init__(s,nc):
        super().__init__(); s.ve=VisionEncoder(D); s.te=TextEncoder(VOCAB_SIZE,D,TEXT_ENC_LAYERS,HEADS,MAX_SEQ,DROP)
        s.vr=nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)]); s.tr=nn.ModuleList([TransformerBlock(D,HEADS,dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)]); s.trn=nn.LayerNorm(D)
        s.qa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.qg=nn.Linear(D,D); s.qn=nn.LayerNorm(D)
        s.fl=nn.ModuleList([FusionLayer(D,HEADS,DROP) for _ in range(FUSE_LAYERS)])
        s.pq=nn.Parameter(torch.randn(1,1,D)*0.02); s.pa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.pn=nn.LayerNorm(D)
        s.h1=nn.Linear(D,D); s.d1=nn.Dropout(DROP); s.h2=nn.Linear(D,D//2); s.d2=nn.Dropout(DROP); s.ho=nn.Linear(D//2,nc); s.hr=nn.Linear(D,D//2); s.hn=nn.LayerNorm(D//2)
    def forward(s,img,ids,mask):
        v=s.ve(img); t=s.te(ids,mask=mask)
        for b in s.vr: v=b(v)
        for b in s.tr: t=b(t,mask=mask)
        t=s.trn(t); qc=t[:,0:1,:].expand(-1,v.shape[1],-1); ao,_=s.qa(qc,v,v); g=torch.sigmoid(s.qg(ao)); v=s.qn(v+v*g+ao*(1-g))
        kpm=(mask==0)
        for f in s.fl: v,t=f(v,t,kpm=kpm)
        c=torch.cat([v,t],1); B=c.shape[0]; pq=s.pq.expand(B,-1,-1); p,_=s.pa(pq,c,c); f=s.pn(pq+p).squeeze(1)
        h=s.d1(F.gelu(s.h1(f))); h=s.d2(F.gelu(s.h2(h))); return s.ho(s.hn(h+s.hr(f)))

gm=MedicalVQAModel(num_classes).to(device); n_params=sum(p.numel() for p in gm.parameters()); print(f"\n  Model: {n_params:,} ({n_params/1e6:.1f}M)")

# ── FedAvg ──
print("\n"+"="*60+"\nTRAINING (FedAvg)\n"+"="*60)
def get_p(m): return [p.data.cpu().numpy().copy() for p in m.parameters()]
def set_p(m,ps):
    for p,w in zip(m.parameters(),ps): p.data=torch.from_numpy(w).to(p.device)
def agg(cp,sizes):
    total=sum(sizes); wts=[n/total for n in sizes]; return [sum(wts[i]*cp[i][p] for i in range(len(cp))) for p in range(len(cp[0]))]

criterion=nn.CrossEntropyLoss()
@torch.no_grad()
def ev(loader):
    gm.eval(); ls,c,t=0.0,0,0
    for i,d,m,l in loader: i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device); lo=gm(i,d,m); ls+=criterion(lo,l).item()*l.size(0); c+=(lo.argmax(-1)==l).sum().item(); t+=l.size(0)
    return ls/t,100*c/t

history={'round':[],'avg_train_loss':[],'avg_train_acc':[],'test_loss':[],'test_acc':[],'round_time':[]}
for cid in range(NUM_CLIENTS): history[f'c{cid}_loss']=[]; history[f'c{cid}_acc']=[]
best_acc,best_state=0.0,None

for rnd in range(1,ROUNDS+1):
    t0=time.time(); gp=get_p(gm); rcp=[]; rl=[]; rc,rt=0,0
    pb=tqdm(range(NUM_CLIENTS),desc=f"R{rnd:02d}/{ROUNDS}",leave=False)
    for cid in pb:
        local=copy.deepcopy(gm); set_p(local,[p.copy() for p in gp]); local.train()
        opt=torch.optim.AdamW(local.parameters(),lr=FED_LR,weight_decay=WD); cl,cc,ct=0.0,0,0
        for _ in range(LOCAL_EP):
            for i,d,m,l in client_loaders[cid]:
                i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device); opt.zero_grad(); lo=local(i,d,m); loss=criterion(lo,l); loss.backward()
                nn.utils.clip_grad_norm_(local.parameters(),1.0); opt.step(); cl+=loss.item()*l.size(0); cc+=(lo.argmax(-1)==l).sum().item(); ct+=l.size(0)
        rcp.append(get_p(local)); c_l=cl/max(ct,1); c_a=100*cc/max(ct,1); rl.append(c_l); rc+=cc; rt+=ct
        history[f'c{cid}_loss'].append(round(c_l,4)); history[f'c{cid}_acc'].append(round(c_a,2)); pb.set_postfix(C=cid,l=f"{c_l:.3f}"); del local,opt
    set_p(gm,agg(rcp,list(client_sizes.values()))); tel,tea=ev(test_loader); rtime=time.time()-t0
    al=np.mean(rl); aa=100*rc/max(rt,1)
    history['round'].append(rnd); history['avg_train_loss'].append(round(al,4)); history['avg_train_acc'].append(round(aa,2))
    history['test_loss'].append(round(tel,4)); history['test_acc'].append(round(tea,2)); history['round_time'].append(round(rtime,1))
    mk=""
    if tea>best_acc: best_acc=tea; best_state=copy.deepcopy(gm.state_dict()); mk=" ★"
    print(f"R{rnd:02d} [{rtime:.1f}s]  Train: {al:.4f}/{aa:.1f}%  Test: {tel:.4f}/{tea:.1f}%{mk}")

if best_state: gm.load_state_dict(best_state)
tel,tea=ev(test_loader); print(f"\n{'='*60}\nFINAL: {tea:.2f}%\n{'='*60}")

# ── SAVE EXCEL ──
import openpyxl; from openpyxl.styles import Font,PatternFill,Alignment
wb=openpyxl.Workbook(); ws=wb.active; ws.title="Training"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='1A5276',end_color='1A5276',fill_type='solid')
headers=['Round','Avg Train Loss','Avg Train Acc (%)','Test Loss','Test Acc (%)','Time (s)']
for cid in range(NUM_CLIENTS): headers+=[f'C{cid} Loss',f'C{cid} Acc (%)']
for c,h in enumerate(headers,1): cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i,rnd in enumerate(history['round']):
    r=i+2; ws.cell(row=r,column=1,value=rnd); ws.cell(row=r,column=2,value=history['avg_train_loss'][i]); ws.cell(row=r,column=3,value=history['avg_train_acc'][i])
    ws.cell(row=r,column=4,value=history['test_loss'][i]); ws.cell(row=r,column=5,value=history['test_acc'][i]); ws.cell(row=r,column=6,value=history['round_time'][i])
    for cid in range(NUM_CLIENTS): ws.cell(row=r,column=7+cid*2,value=history[f'c{cid}_loss'][i]); ws.cell(row=r,column=8+cid*2,value=history[f'c{cid}_acc'][i])
ws2=wb.create_sheet("Summary")
for i,(k,v) in enumerate([("Method","FedAvg"),("Dataset","VizWiz"),("Params",f"{n_params:,}"),("Clients",NUM_CLIENTS),("Rounds",ROUNDS),("Best Test",round(best_acc,2)),("Final Test",round(tea,2))],1):
    ws2.cell(row=i,column=1,value=k).font=Font(bold=True,name='Arial'); ws2.cell(row=i,column=2,value=v)
for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/vizwiz_fedavg_custom_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")

Device: cuda
GPU: Tesla T4, VRAM: 15.6 GB

LOADING VizWiz INTO RAM


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00008-8bb04d0d8c47d4a(…):   0%|          | 0.00/494M [00:00<?, ?B/s]

data/test-00001-of-00008-0deaf01822797f0(…):   0%|          | 0.00/530M [00:00<?, ?B/s]

data/test-00002-of-00008-a4468dc23224a35(…):   0%|          | 0.00/482M [00:00<?, ?B/s]

...

R18 [33.4s]  Train: 1.0702/76.2%  Test: 2.6614/59.3%


R19/20:   0%|          | 0/5 [00:00<?, ?it/s]

R19 [33.3s]  Train: 0.9901/78.4%  Test: 2.6176/58.5%


R20/20:   0%|          | 0/5 [00:00<?, ?it/s]

R20 [33.2s]  Train: 0.9134/80.4%  Test: 2.7194/60.0%

FINAL: 63.49%

Saved → /kaggle/working//vizwiz_fedavg_custom_results.xlsx
DONE!
